# Unsupervised Learning: Clustering Methods
**Prepared by: Harsahib Singh.**

**Lab: Machine Learning.**

This notebook covers unsupervised learning techniques including K-Means clustering, Elbow Method, Silhouette Analysis, and Gaussian Mixture Models (GMM) implemented using PyTorch and scikit-learn.

## 1. Import Required Libraries

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons, load_iris, load_digits
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch Version: {torch.__version__}')

## 2. Unsupervised Learning Theory

### What is Clustering?
Grouping similar data points together without labeled data

### Key Concepts
- **Intra-cluster distance**: Distance within a cluster (minimize)
- **Inter-cluster distance**: Distance between clusters (maximize)

### Clustering Methods
1. **Partitioning**: K-Means, K-Medoids
2. **Hierarchical**: Agglomerative, Divisive
3. **Density-based**: DBSCAN, OPTICS
4. **Model-based**: Gaussian Mixture Models (GMM)

## 3. Generate Synthetic Dataset

In [ ]:
# Generate blob data
np.random.seed(42)
X_blobs, y_true = make_blobs(n_samples=400, centers=4, n_features=2,
                              cluster_std=1.0, random_state=42)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true, cmap='viridis',
           s=100, alpha=0.6, edgecolor='black')
plt.title('Synthetic Dataset (True Labels)', fontsize=14, fontweight='bold')
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.colorbar(label='True Cluster')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Dataset shape: {X_blobs.shape}')
print(f'True number of clusters: {len(np.unique(y_true))}')

## 4. K-Means Clustering - PyTorch Implementation

In [ ]:
class KMeans_PyTorch:
    def __init__(self, n_clusters=3, max_iters=100, tol=1e-4):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.tol = tol
        self.centroids = None
        self.labels = None
        self.inertia_ = None

    def fit(self, X):
        # Initialize centroids randomly
        n_samples = X.shape[0]
        random_indices = torch.randperm(n_samples)[:self.n_clusters]
        self.centroids = X[random_indices].clone()

        for iteration in range(self.max_iters):
            # Assign points to nearest centroid
            distances = torch.cdist(X, self.centroids)
            self.labels = torch.argmin(distances, dim=1)

            # Store old centroids
            old_centroids = self.centroids.clone()

            # Update centroids
            for k in range(self.n_clusters):
                cluster_points = X[self.labels == k]
                if len(cluster_points) > 0:
                    self.centroids[k] = torch.mean(cluster_points, dim=0)

            # Check convergence
            centroid_shift = torch.sum(torch.sqrt(torch.sum((self.centroids - old_centroids) ** 2, dim=1)))
            if centroid_shift < self.tol:
                print(f'Converged at iteration {iteration + 1}')
                break

        # Calculate inertia (within-cluster sum of squares)
        distances = torch.cdist(X, self.centroids)
        self.inertia_ = torch.sum(torch.min(distances, dim=1)[0] ** 2).item()

        return self

    def predict(self, X):
        distances = torch.cdist(X, self.centroids)
        return torch.argmin(distances, dim=1)

print('K-Means PyTorch class implemented!')

In [ ]:
# Convert to PyTorch tensors
X_blobs_tensor = torch.FloatTensor(X_blobs)

# Train K-Means
kmeans_torch = KMeans_PyTorch(n_clusters=4, max_iters=100)
kmeans_torch.fit(X_blobs_tensor)

# Visualize results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true, cmap='viridis',
           s=100, alpha=0.6, edgecolor='black')
plt.title('True Labels', fontsize=13, fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 2, 2)
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans_torch.labels.numpy(),
           cmap='viridis', s=100, alpha=0.6, edgecolor='black')
plt.scatter(kmeans_torch.centroids[:, 0].numpy(), kmeans_torch.centroids[:, 1].numpy(),
           c='red', s=300, marker='X', edgecolor='black', linewidth=2, label='Centroids')
plt.title('K-Means Clustering (PyTorch)', fontsize=13, fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()

plt.tight_layout()
plt.show()

print(f'Inertia: {kmeans_torch.inertia_:.2f}')

## 5. K-Means Algorithm Visualization (Step-by-Step)

In [ ]:
# Visualize K-Means iterations
def visualize_kmeans_steps(X, n_clusters=4, max_steps=5):
    X_tensor = torch.FloatTensor(X)
    n_samples = X_tensor.shape[0]

    # Initialize centroids
    random_indices = torch.randperm(n_samples)[:n_clusters]
    centroids = X_tensor[random_indices].clone()

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.ravel()

    for step in range(max_steps):
        # Assign clusters
        distances = torch.cdist(X_tensor, centroids)
        labels = torch.argmin(distances, dim=1)

        # Plot
        axes[step].scatter(X[:, 0], X[:, 1], c=labels.numpy(), cmap='viridis',
                          s=100, alpha=0.6, edgecolor='black')
        axes[step].scatter(centroids[:, 0].numpy(), centroids[:, 1].numpy(),
                          c='red', s=300, marker='X', edgecolor='black', linewidth=2)
        axes[step].set_title(f'Iteration {step + 1}', fontsize=12, fontweight='bold')
        axes[step].set_xlabel('Feature 1')
        axes[step].set_ylabel('Feature 2')

        # Update centroids
        for k in range(n_clusters):
            cluster_points = X_tensor[labels == k]
            if len(cluster_points) > 0:
                centroids[k] = torch.mean(cluster_points, dim=0)

    # Remove extra subplot
    fig.delaxes(axes[5])
    plt.tight_layout()
    plt.show()

visualize_kmeans_steps(X_blobs, n_clusters=4, max_steps=5)

## 6. Elbow Method - Finding Optimal K

In [ ]:
# Calculate inertia for different k values
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans_sk = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_sk.fit(X_blobs)
    inertias.append(kmeans_sk.inertia_)
    silhouette_scores.append(silhouette_score(X_blobs, kmeans_sk.labels_))

# Plot Elbow curve
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(k_range, inertias, marker='o', linewidth=2, markersize=10)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].axvline(x=4, color='red', linestyle='--', linewidth=2, label='Optimal k=4')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(k_range, silhouette_scores, marker='s', linewidth=2, markersize=10, color='green')
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs K', fontsize=14, fontweight='bold')
axes[1].axvline(x=4, color='red', linestyle='--', linewidth=2, label='Optimal k=4')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

optimal_k = list(k_range)[np.argmax(silhouette_scores)]
print(f'Optimal k by Silhouette Score: {optimal_k}')
print(f'Best Silhouette Score: {max(silhouette_scores):.4f}')

## 7. Silhouette Analysis in Detail

In [ ]:
# Detailed silhouette analysis for k=4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_blobs)

# Calculate silhouette scores
silhouette_avg = silhouette_score(X_blobs, cluster_labels)
sample_silhouette_values = silhouette_samples(X_blobs, cluster_labels)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Silhouette plot
y_lower = 10
for i in range(4):
    cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]
    cluster_silhouette_values.sort()

    size_cluster_i = cluster_silhouette_values.shape[0]
    y_upper = y_lower + size_cluster_i

    color = plt.cm.viridis(float(i) / 4)
    ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_values,
                      facecolor=color, edgecolor=color, alpha=0.7)

    ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
    y_lower = y_upper + 10

ax1.set_xlabel('Silhouette Coefficient', fontsize=12)
ax1.set_ylabel('Cluster', fontsize=12)
ax1.set_title('Silhouette Plot for Each Cluster', fontsize=13, fontweight='bold')
ax1.axvline(x=silhouette_avg, color='red', linestyle='--', linewidth=2,
           label=f'Average Score: {silhouette_avg:.3f}')
ax1.set_yticks([])
ax1.legend()

# Cluster visualization
colors = plt.cm.viridis(cluster_labels.astype(float) / 4)
ax2.scatter(X_blobs[:, 0], X_blobs[:, 1], c=colors, s=100, alpha=0.6, edgecolor='black')
ax2.scatter(kmeans_final.cluster_centers_[:, 0], kmeans_final.cluster_centers_[:, 1],
           c='red', s=300, marker='X', edgecolor='black', linewidth=2, label='Centroids')
ax2.set_title('K-Means Clustering Result', fontsize=13, fontweight='bold')
ax2.set_xlabel('Feature 1', fontsize=12)
ax2.set_ylabel('Feature 2', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Average Silhouette Score: {silhouette_avg:.4f}')
print(f'Davies-Bouldin Index: {davies_bouldin_score(X_blobs, cluster_labels):.4f} (lower is better)')

## 8. Gaussian Mixture Model (GMM)

In [ ]:
# Train GMM
gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
gmm.fit(X_blobs)

# Predict clusters
gmm_labels = gmm.predict(X_blobs)
gmm_probs = gmm.predict_proba(X_blobs)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Hard assignment
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=gmm_labels, cmap='viridis',
               s=100, alpha=0.6, edgecolor='black')
axes[0].scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', s=300,
               marker='X', edgecolor='black', linewidth=2, label='Means')
axes[0].set_title('GMM - Hard Assignment', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature 1', fontsize=12)
axes[0].set_ylabel('Feature 2', fontsize=12)
axes[0].legend()

# Soft assignment (probability)
max_probs = np.max(gmm_probs, axis=1)
scatter = axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=gmm_labels,
                         cmap='viridis', s=100, alpha=max_probs, edgecolor='black')
axes[1].scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', s=300,
               marker='X', edgecolor='black', linewidth=2, label='Means')
axes[1].set_title('GMM - Soft Assignment (Opacity = Confidence)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature 1', fontsize=12)
axes[1].set_ylabel('Feature 2', fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'GMM Log-Likelihood: {gmm.score(X_blobs):.2f}')
print(f'BIC: {gmm.bic(X_blobs):.2f}')
print(f'AIC: {gmm.aic(X_blobs):.2f}')

## 9. GMM Model Selection (BIC and AIC)

In [ ]:
# Test different numbers of components
n_components_range = range(2, 11)
bic_scores = []
aic_scores = []
log_likelihoods = []

for n_comp in n_components_range:
    gmm_temp = GaussianMixture(n_components=n_comp, covariance_type='full', random_state=42)
    gmm_temp.fit(X_blobs)
    bic_scores.append(gmm_temp.bic(X_blobs))
    aic_scores.append(gmm_temp.aic(X_blobs))
    log_likelihoods.append(gmm_temp.score(X_blobs))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(n_components_range, bic_scores, marker='o', linewidth=2, label='BIC', markersize=8)
axes[0].plot(n_components_range, aic_scores, marker='s', linewidth=2, label='AIC', markersize=8)
axes[0].set_xlabel('Number of Components', fontsize=12)
axes[0].set_ylabel('Information Criterion', fontsize=12)
axes[0].set_title('Model Selection: BIC and AIC (Lower is Better)', fontsize=13, fontweight='bold')
axes[0].axvline(x=4, color='red', linestyle='--', linewidth=2, label='True k=4')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(n_components_range, log_likelihoods, marker='D', linewidth=2,
            color='green', markersize=8)
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Log-Likelihood', fontsize=12)
axes[1].set_title('Log-Likelihood vs Components (Higher is Better)', fontsize=13, fontweight='bold')
axes[1].axvline(x=4, color='red', linestyle='--', linewidth=2, label='True k=4')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

optimal_n_components_bic = list(n_components_range)[np.argmin(bic_scores)]
optimal_n_components_aic = list(n_components_range)[np.argmin(aic_scores)]
print(f'Optimal components by BIC: {optimal_n_components_bic}')
print(f'Optimal components by AIC: {optimal_n_components_aic}')

## 10. K-Means vs GMM Comparison

In [ ]:
# Compare K-Means and GMM
kmeans_comp = KMeans(n_clusters=4, random_state=42)
kmeans_labels_comp = kmeans_comp.fit_predict(X_blobs)

gmm_comp = GaussianMixture(n_components=4, random_state=42)
gmm_labels_comp = gmm_comp.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# True labels
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true, cmap='viridis',
               s=100, alpha=0.6, edgecolor='black')
axes[0].set_title('True Labels', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# K-Means
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans_labels_comp, cmap='viridis',
               s=100, alpha=0.6, edgecolor='black')
axes[1].scatter(kmeans_comp.cluster_centers_[:, 0], kmeans_comp.cluster_centers_[:, 1],
               c='red', s=300, marker='X', edgecolor='black', linewidth=2)
km_silhouette = silhouette_score(X_blobs, kmeans_labels_comp)
axes[1].set_title(f'K-Means (Silhouette: {km_silhouette:.3f})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

# GMM
axes[2].scatter(X_blobs[:, 0], X_blobs[:, 1], c=gmm_labels_comp, cmap='viridis',
               s=100, alpha=0.6, edgecolor='black')
axes[2].scatter(gmm_comp.means_[:, 0], gmm_comp.means_[:, 1],
               c='red', s=300, marker='X', edgecolor='black', linewidth=2)
gmm_silhouette = silhouette_score(X_blobs, gmm_labels_comp)
axes[2].set_title(f'GMM (Silhouette: {gmm_silhouette:.3f})', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Feature 1')
axes[2].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

# Comparison metrics
print('\nComparison Metrics:')
print(f'K-Means Silhouette Score: {km_silhouette:.4f}')
print(f'GMM Silhouette Score: {gmm_silhouette:.4f}')
print(f'K-Means ARI: {adjusted_rand_score(y_true, kmeans_labels_comp):.4f}')
print(f'GMM ARI: {adjusted_rand_score(y_true, gmm_labels_comp):.4f}')
print(f'K-Means NMI: {normalized_mutual_info_score(y_true, kmeans_labels_comp):.4f}')
print(f'GMM NMI: {normalized_mutual_info_score(y_true, gmm_labels_comp):.4f}')

## 11. Real-World Application - Iris Dataset

In [ ]:
# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris_true = iris.target

# Standardize
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Apply PCA for visualization
pca = PCA(n_components=2)
X_iris_pca = pca.fit_transform(X_iris_scaled)

print(f'Explained variance ratio: {pca.explained_variance_ratio_}')
print(f'Total variance explained: {sum(pca.explained_variance_ratio_):.2%}')

In [ ]:
# Cluster Iris data
kmeans_iris = KMeans(n_clusters=3, random_state=42)
kmeans_iris_labels = kmeans_iris.fit_predict(X_iris_scaled)

gmm_iris = GaussianMixture(n_components=3, random_state=42)
gmm_iris_labels = gmm_iris.fit_predict(X_iris_scaled)

# Visualize in PCA space
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# True labels
scatter1 = axes[0].scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=y_iris_true,
                          cmap='viridis', s=100, alpha=0.6, edgecolor='black')
axes[0].set_title('True Species Labels', fontsize=13, fontweight='bold')
axes[0].set_xlabel('First Principal Component')
axes[0].set_ylabel('Second Principal Component')
plt.colorbar(scatter1, ax=axes[0], label='Species')

# K-Means
scatter2 = axes[1].scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=kmeans_iris_labels,
                          cmap='viridis', s=100, alpha=0.6, edgecolor='black')
axes[1].set_title(f'K-Means Clustering (ARI: {adjusted_rand_score(y_iris_true, kmeans_iris_labels):.3f})',
                 fontsize=13, fontweight='bold')
axes[1].set_xlabel('First Principal Component')
axes[1].set_ylabel('Second Principal Component')

# GMM
scatter3 = axes[2].scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=gmm_iris_labels,
                          cmap='viridis', s=100, alpha=0.6, edgecolor='black')
axes[2].set_title(f'GMM Clustering (ARI: {adjusted_rand_score(y_iris_true, gmm_iris_labels):.3f})',
                 fontsize=13, fontweight='bold')
axes[2].set_xlabel('First Principal Component')
axes[2].set_ylabel('Second Principal Component')

plt.tight_layout()
plt.show()

print('\nClustering Performance on Iris Dataset:')
print(f'K-Means Adjusted Rand Index: {adjusted_rand_score(y_iris_true, kmeans_iris_labels):.4f}')
print(f'GMM Adjusted Rand Index: {adjusted_rand_score(y_iris_true, gmm_iris_labels):.4f}')
print(f'K-Means Silhouette Score: {silhouette_score(X_iris_scaled, kmeans_iris_labels):.4f}')
print(f'GMM Silhouette Score: {silhouette_score(X_iris_scaled, gmm_iris_labels):.4f}')

## 12. Different Covariance Types in GMM

In [ ]:
# Test different covariance types
covariance_types = ['full', 'tied', 'diag', 'spherical']
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, cov_type in enumerate(covariance_types):
    gmm_cov = GaussianMixture(n_components=4, covariance_type=cov_type, random_state=42)
    gmm_cov.fit(X_blobs)
    labels_cov = gmm_cov.predict(X_blobs)

    bic = gmm_cov.bic(X_blobs)
    silhouette = silhouette_score(X_blobs, labels_cov)

    axes[idx].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels_cov, cmap='viridis',
                     s=100, alpha=0.6, edgecolor='black')
    axes[idx].scatter(gmm_cov.means_[:, 0], gmm_cov.means_[:, 1], c='red',
                     s=300, marker='X', edgecolor='black', linewidth=2)
    axes[idx].set_title(f'{cov_type.upper()} (BIC: {bic:.0f}, Silhouette: {silhouette:.3f})',
                       fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print('Covariance Types:')
print('- Full: Each component has its own covariance matrix')
print('- Tied: All components share the same covariance matrix')
print('- Diag: Diagonal covariance (features independent within cluster)')
print('- Spherical: Spherical covariance (all features have same variance)')

## 13. Key Takeaways

### K-Means Clustering
**Algorithm:**
1. Initialize k centroids randomly
2. Assign each point to nearest centroid
3. Update centroids as mean of assigned points
4. Repeat until convergence

**Advantages:**
- Simple and fast
- Works well with spherical clusters
- Scales to large datasets

**Limitations:**
- Requires specifying k
- Sensitive to initialization
- Assumes spherical clusters of similar size
- Hard assignment (no uncertainty)

### Elbow Method
- Plot inertia vs number of clusters
- Look for elbow (point of diminishing returns)
- Not always clear

### Silhouette Analysis
**Silhouette Coefficient:**
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

Where:
- $a(i)$: Mean distance to points in same cluster
- $b(i)$: Mean distance to points in nearest cluster
- Range: [-1, 1], higher is better

**Interpretation:**
- Close to +1: Well-clustered
- Close to 0: On boundary
- Negative: Possibly wrong cluster

### Gaussian Mixture Models (GMM)
**Key Features:**
- Probabilistic model
- Soft assignment (membership probabilities)
- Allows elliptical clusters
- Uses Expectation-Maximization (EM) algorithm

**Advantages over K-Means:**
- Provides uncertainty estimates
- More flexible cluster shapes
- Statistical foundation

**Model Selection:**
- **BIC** (Bayesian Information Criterion): Penalizes complexity more
- **AIC** (Akaike Information Criterion): Less penalty for complexity
- Both: Lower is better

### When to Use What
- **K-Means**: Fast, simple, spherical clusters, large datasets
- **GMM**: Need probabilities, elliptical clusters, smaller datasets
- **Silhouette**: Evaluate clustering quality
- **Elbow + Silhouette**: Find optimal number of clusters

### Evaluation Metrics
**When true labels available:**
- Adjusted Rand Index (ARI)
- Normalized Mutual Information (NMI)

**When true labels unavailable:**
- Silhouette Score
- Davies-Bouldin Index
- Calinski-Harabasz Index